# 02 · 多分位风险模型与因果校准

本阶段把 `direct_lgbm` 与 `shape_strength` 的每个 quantile head 视为独立 operating-point candidate。模型报告整体与 `side × ratio_bucket` coverage、pinball、final_month 评价和 crossing_rate，但不强制跨 quantile 排序。

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'outputs' / '01_stage_contract.json').exists():
            return candidate
    raise FileNotFoundError('run 01_build_panel_and_labels.ipynb first')


def quantile_label(quantile: float) -> str:
    return f'q{int(round(100 * quantile)):02d}'


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'outputs'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
STAGE = json.loads((OUTPUT_DIR / '01_stage_contract.json').read_text(encoding='utf-8'))
RUN_MODE = STAGE['run_mode']
QUANTILE_REGISTRY = [0.50, 0.80, 0.85, 0.90, 0.95, 0.99]
RAW_TAU_MAP = {0.50: 0.50, 0.80: 0.80, 0.85: 0.85, 0.90: 0.90, 0.95: 0.80, 0.99: 0.90}
IMPACT_GUARDRAIL_QUANTILE = 0.95
RANDOM_SEED = 20260807
model_panel = pd.read_parquet(ROOT / STAGE['model_panel'])
model_panel['date'] = pd.to_datetime(model_panel['date'], errors='coerce').dt.normalize()
feature_columns = list(STAGE['feature_columns'])


In [ ]:
unique_dates = sorted(model_panel['date'].dropna().unique())
if len(unique_dates) < 10:
    raise ValueError('at least 10 trading dates are required')
n_test_dates = max(1, math.ceil(len(unique_dates) * 0.20))
n_calibration_dates = max(1, math.ceil(len(unique_dates) * 0.10))
train_dates = set(unique_dates[:-(n_test_dates + n_calibration_dates)])
calibration_dates = set(unique_dates[-(n_test_dates + n_calibration_dates):-n_test_dates])
test_dates = set(unique_dates[-n_test_dates:])
if train_dates & calibration_dates or train_dates & test_dates or calibration_dates & test_dates:
    raise ValueError('time split overlap detected')
train = model_panel[model_panel['date'].isin(train_dates)].copy()
calibration = model_panel[model_panel['date'].isin(calibration_dates)].copy()
test = model_panel[model_panel['date'].isin(test_dates)].copy()

medians = train[feature_columns].apply(pd.to_numeric, errors='coerce').median().fillna(0.0).to_dict()


def make_matrix(frame: pd.DataFrame, columns: list[str] = feature_columns) -> pd.DataFrame:
    matrix = frame.reindex(columns=columns).apply(pd.to_numeric, errors='coerce')
    for column in columns:
        matrix[column] = matrix[column].fillna(float(medians.get(column, 0.0)))
    return matrix.astype('float32')


split_diagnostics = pd.DataFrame([{
    'n_dates': len(unique_dates),
    'train_min_date': min(train_dates), 'train_max_date': max(train_dates),
    'calibration_min_date': min(calibration_dates), 'calibration_max_date': max(calibration_dates),
    'test_min_date': min(test_dates), 'test_max_date': max(test_dates),
    'n_train': len(train), 'n_calibration': len(calibration), 'n_test': len(test),
    'final_month_learning_excluded': True,
}])
split_diagnostics.to_csv(OUTPUT_DIR / '02_time_split.csv', index=False)
print(split_diagnostics.to_string(index=False))


In [ ]:
def model_parameters(alpha: float) -> dict:
    return {
        'objective': 'quantile', 'alpha': float(alpha),
        'n_estimators': 45 if RUN_MODE == 'smoke' else 280,
        'learning_rate': 0.055, 'num_leaves': 31, 'min_child_samples': 80,
        'subsample': 0.90, 'colsample_bytree': 0.90,
        'reg_lambda': 1.0, 'random_state': RANDOM_SEED,
        'n_jobs': 1 if RUN_MODE == 'smoke' else -1,
        'deterministic': True, 'force_col_wise': True, 'verbosity': -1,
    }


X_train = make_matrix(train)
y_total_train = pd.to_numeric(train['total_bad_move_bps'], errors='coerce').fillna(0.0)
y_impact_train = pd.to_numeric(train['impact_me_bad_bps'], errors='coerce').fillna(0.0)
direct_models = {}
for quantile in QUANTILE_REGISTRY:
    model = LGBMRegressor(**model_parameters(RAW_TAU_MAP[quantile]))
    model.fit(X_train, y_total_train)
    direct_models[quantile] = model
impact_model = LGBMRegressor(**model_parameters(IMPACT_GUARDRAIL_QUANTILE))
impact_model.fit(X_train, y_impact_train)

condition_features = [column for column in feature_columns if column != 'x_adv']
X_condition_train = make_matrix(train, condition_features)
shape_total_train = np.maximum(y_total_train.to_numpy(), 0.0)
shape_models = {}
shape_grids = {}
for quantile in QUANTILE_REGISTRY:
    shape_frame = pd.DataFrame({'x_adv': train['x_adv'].to_numpy(), 'shape_total': shape_total_train})
    h_grid = shape_frame.groupby('x_adv', as_index=False)['shape_total'].quantile(quantile).sort_values('x_adv')
    h_grid['h_value'] = np.maximum.accumulate(np.maximum(h_grid['shape_total'].to_numpy(dtype=float), 0.0))
    scale = float(h_grid['h_value'].max())
    if not np.isfinite(scale) or scale <= 1e-9:
        h_grid['h_value'] = h_grid['x_adv'] / max(float(h_grid['x_adv'].max()), 1e-9)
    else:
        h_grid['h_value'] = h_grid['h_value'] / scale
    h_values = np.interp(train['x_adv'], h_grid['x_adv'], h_grid['h_value'], left=0.0, right=1.0)
    strength_target = shape_total_train / np.maximum(h_values, 0.05)
    strength_cap = float(np.nanquantile(strength_target, 0.995))
    strength_target = np.clip(strength_target, 0.0, max(strength_cap, 1.0))
    strength_model = LGBMRegressor(**model_parameters(quantile))
    strength_model.fit(X_condition_train, strength_target)
    shape_models[quantile] = strength_model
    shape_grids[quantile] = h_grid[['x_adv', 'h_value']].reset_index(drop=True)


In [ ]:
prediction_keys = ['date', 'sym', 'side', 'quote_strategy', 'x_adv', 'ratio_bucket', 'total_bad_move_bps', 'impact_me_bad_bps']
prediction_frame = pd.concat([
    calibration[prediction_keys].assign(split='calibration'),
    test[prediction_keys].assign(split='test'),
], ignore_index=True)
X_prediction = make_matrix(prediction_frame)
X_condition_prediction = make_matrix(prediction_frame, condition_features)
for quantile in QUANTILE_REGISTRY:
    qn = quantile_label(quantile)
    prediction_frame[f'direct_lgbm_{qn}_raw'] = np.maximum(direct_models[quantile].predict(X_prediction), 0.0)
    h_grid = shape_grids[quantile]
    h_values = np.interp(prediction_frame['x_adv'], h_grid['x_adv'], h_grid['h_value'], left=0.0, right=1.0)
    strength = np.maximum(shape_models[quantile].predict(X_condition_prediction), 0.0)
    prediction_frame[f'shape_strength_{qn}_raw'] = strength * h_values
prediction_frame['impact_guardrail_q95_raw'] = np.maximum(impact_model.predict(X_prediction), 0.0)


def target_coverage(quantile: float) -> float:
    return 0.955 if np.isclose(quantile, 0.95) else float(quantile)


def residual_buffer(values: pd.Series, target: float) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    return max(float(clean.quantile(target)), 0.0) if len(clean) else 0.0


calibration_rows = []
cal_mask = prediction_frame['split'].eq('calibration')
calibration_date_values = sorted(prediction_frame.loc[cal_mask, 'date'].dropna().unique())
fold_edges = np.array_split(calibration_date_values, min(3, len(calibration_date_values)))
for family in ['direct_lgbm', 'shape_strength']:
    for quantile in QUANTILE_REGISTRY:
        qn = quantile_label(quantile)
        raw_col = f'{family}_{qn}_raw'
        residual = prediction_frame.loc[cal_mask, 'total_bad_move_bps'] - prediction_frame.loc[cal_mask, raw_col]
        global_buffer = residual_buffer(residual, target_coverage(quantile))
        causal_history = []
        prior_dates = []
        for fold_dates in fold_edges:
            if prior_dates:
                prior_mask = cal_mask & prediction_frame['date'].isin(prior_dates)
                prior_residual = prediction_frame.loc[prior_mask, 'total_bad_move_bps'] - prediction_frame.loc[prior_mask, raw_col]
                causal_history.append(residual_buffer(prior_residual, target_coverage(quantile)))
            prior_dates.extend(list(fold_dates))
        causal_floor = float(np.quantile(causal_history, 0.80)) if causal_history else 0.0
        for (side, bucket), index in prediction_frame.loc[cal_mask].groupby(['side', 'ratio_bucket']).groups.items():
            bucket_residual = prediction_frame.loc[index, 'total_bad_move_bps'] - prediction_frame.loc[index, raw_col]
            bucket_buffer = residual_buffer(bucket_residual, target_coverage(quantile)) if len(index) >= 20 else global_buffer
            calibration_rows.append({
                'model_family': family, 'quantile_level': quantile, 'side': side, 'ratio_bucket': bucket,
                'n': len(index), 'target_coverage': target_coverage(quantile),
                'calibration_buffer_bps': bucket_buffer, 'causal_floor_buffer_bps': causal_floor,
                'final_buffer_bps': max(bucket_buffer, causal_floor),
                'rolling_floor_final_month_excluded': True,
            })
        buffer_map = {(row['side'], row['ratio_bucket']): row for row in calibration_rows if row['model_family'] == family and np.isclose(row['quantile_level'], quantile)}
        buffers = [buffer_map.get((side, bucket), {'calibration_buffer_bps': global_buffer, 'final_buffer_bps': max(global_buffer, causal_floor)}) for side, bucket in zip(prediction_frame['side'], prediction_frame['ratio_bucket'])]
        prediction_frame[f'{family}_{qn}_calibrated'] = prediction_frame[raw_col] + np.array([row['calibration_buffer_bps'] for row in buffers])
        prediction_frame[f'{family}_{qn}_final'] = prediction_frame[raw_col] + np.array([row['final_buffer_bps'] for row in buffers])

impact_residual = prediction_frame.loc[cal_mask, 'impact_me_bad_bps'] - prediction_frame.loc[cal_mask, 'impact_guardrail_q95_raw']
impact_buffer = residual_buffer(impact_residual, IMPACT_GUARDRAIL_QUANTILE)
prediction_frame['impact_guardrail_q95_final'] = prediction_frame['impact_guardrail_q95_raw'] + impact_buffer
calibration_table = pd.DataFrame(calibration_rows)
calibration_table.to_csv(OUTPUT_DIR / '02_calibration_table.csv', index=False)


In [ ]:
def pinball_loss(y_true: pd.Series, y_pred: pd.Series, quantile: float) -> float:
    error = pd.to_numeric(y_true, errors='coerce') - pd.to_numeric(y_pred, errors='coerce')
    return float(np.nanmean(np.maximum(quantile * error, (quantile - 1.0) * error)))


metric_rows = []
group_rows = []
for family in ['direct_lgbm', 'shape_strength']:
    for quantile in QUANTILE_REGISTRY:
        qn = quantile_label(quantile)
        for stage in ['raw', 'calibrated', 'final']:
            pred_col = f'{family}_{qn}_{stage}'
            for split_name, mask in {
                'calibration': prediction_frame['split'].eq('calibration'),
                'test': prediction_frame['split'].eq('test'),
                'final_month': prediction_frame['split'].eq('test') & prediction_frame['date'].dt.month.eq(12),
            }.items():
                subset = prediction_frame.loc[mask]
                if subset.empty:
                    continue
                coverage = float((subset['total_bad_move_bps'] <= subset[pred_col]).mean())
                metric_rows.append({
                    'model_family': family, 'quantile_level': quantile, 'prediction_stage': stage,
                    'split': split_name, 'n': len(subset), 'coverage': coverage,
                    'target_coverage': target_coverage(quantile),
                    'pinball_loss': pinball_loss(subset['total_bad_move_bps'], subset[pred_col], quantile),
                    'mean_prediction_bps': float(subset[pred_col].mean()),
                    'final_month_learning_excluded': True,
                })
            final_test = prediction_frame[prediction_frame['split'].eq('test')]
            if stage == 'final':
                for (side, bucket), subset in final_test.groupby(['side', 'ratio_bucket'], dropna=False):
                    group_rows.append({
                        'model_family': family, 'quantile_level': quantile, 'prediction_stage': stage,
                        'side': side, 'ratio_bucket': bucket, 'n': len(subset),
                        'coverage': float((subset['total_bad_move_bps'] <= subset[pred_col]).mean()),
                        'target_coverage': target_coverage(quantile),
                        'pinball_loss': pinball_loss(subset['total_bad_move_bps'], subset[pred_col], quantile),
                    })

crossing_rows = []
for family in ['direct_lgbm', 'shape_strength']:
    for stage in ['raw', 'calibrated', 'final']:
        for lower, upper in zip(QUANTILE_REGISTRY[:-1], QUANTILE_REGISTRY[1:]):
            lower_col = f'{family}_{quantile_label(lower)}_{stage}'
            upper_col = f'{family}_{quantile_label(upper)}_{stage}'
            gap = prediction_frame[lower_col] - prediction_frame[upper_col]
            positive = gap[gap > 0]
            crossing_rows.append({
                'model_family': family, 'prediction_stage': stage,
                'lower_quantile': lower, 'upper_quantile': upper,
                'crossing_rate': float((gap > 0).mean()),
                'mean_crossing_bps': float(positive.mean()) if len(positive) else 0.0,
                'p95_crossing_bps': float(positive.quantile(0.95)) if len(positive) else 0.0,
            })

metrics = pd.DataFrame(metric_rows)
grouped_coverage = pd.DataFrame(group_rows)
crossing_diagnostics = pd.DataFrame(crossing_rows)
metrics.to_csv(OUTPUT_DIR / '02_model_metrics.csv', index=False)
grouped_coverage.to_csv(OUTPUT_DIR / '02_grouped_coverage.csv', index=False)
crossing_diagnostics.to_csv(OUTPUT_DIR / '02_crossing_diagnostics.csv', index=False)
prediction_frame.to_parquet(OUTPUT_DIR / '02_predictions.parquet', index=False, compression='zstd')
bundle = {
    'quantile_registry': QUANTILE_REGISTRY, 'raw_tau_map': RAW_TAU_MAP,
    'impact_guardrail_quantile': IMPACT_GUARDRAIL_QUANTILE,
    'feature_columns': feature_columns, 'condition_features': condition_features, 'medians': medians,
    'direct_models': direct_models, 'impact_model': impact_model,
    'shape_strength_models': shape_models, 'shape_grids': shape_grids,
    'calibration_table': calibration_table, 'impact_buffer_bps': impact_buffer,
    'run_mode': RUN_MODE, 'random_seed': RANDOM_SEED,
    'no_order_anchor_source': 'model_prediction_at_x0',
}
joblib.dump(bundle, MODEL_DIR / '02_model_bundle.joblib')
stage_contract = {
    'run_mode': RUN_MODE, 'evaluation_scope': STAGE['evaluation_scope'],
    'quantile_registry': QUANTILE_REGISTRY, 'raw_tau_map': {str(key): value for key, value in RAW_TAU_MAP.items()},
    'cross_quantile_order_enforced': False, 'final_month_learning_excluded': True,
    'no_order_anchor_source': 'model_prediction_at_x0',
    'model_bundle': 'models/02_model_bundle.joblib', 'predictions': 'outputs/02_predictions.parquet',
}
(OUTPUT_DIR / '02_stage_contract.json').write_text(json.dumps(stage_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(metrics.tail(12).to_string(index=False))
print(crossing_diagnostics.to_string(index=False))
